In [100]:
# generate random data matrix
srand(1)
n,d = 10,4
X = randn(n,d)

# optional: give it linearly dependent columns
X[:,3] = X[:,2]

10-element Array{Float64,1}:
  0.583708 
  0.963272 
  0.458791 
 -0.522337 
  0.408396 
 -0.0504512
 -0.693654 
 -1.77383  
  0.120766 
 -0.757633 

In [101]:
rank(X)

3

In [102]:
U,σ,V = svd(X)

([-0.207024 -0.0564923 -0.108327 0.665665; -0.362829 -0.628472 -0.0423898 -0.552876; … ; -0.140507 -0.157656 0.136306 0.168605; 0.0781198 -0.443832 0.497344 0.186254], [3.90441, 3.35786, 3.03002, 9.67579e-17], [-0.69848 0.246793 0.671728 0.0; -0.502368 -0.0894302 -0.489518 -0.707107; -0.502368 -0.0894302 -0.489518 0.707107; 0.0858952 0.96078 -0.263675 2.15106e-15])

In [103]:
U'*U

4×4 Array{Float64,2}:
  1.0          -1.24522e-16   2.52025e-17   2.03562e-16
 -1.24522e-16   1.0           1.23185e-17   1.33246e-16
  2.52025e-17   1.23185e-17   1.0          -3.13176e-16
  2.03562e-16   1.33246e-16  -3.13176e-16   1.0        

In [104]:
V'*V

4×4 Array{Float64,2}:
  1.0           1.00519e-16  -1.42545e-16  -1.76707e-17
  1.00519e-16   1.0           4.89305e-17  -2.23442e-17
 -1.42545e-16   4.89305e-17   1.0          -2.46754e-17
 -1.76707e-17  -2.23442e-17  -2.46754e-17   1.0        

In [105]:
σ

4-element Array{Float64,1}:
 3.90441    
 3.35786    
 3.03002    
 9.67579e-17

In [106]:
Σ = diagm(σ)

4×4 Array{Float64,2}:
 3.90441  0.0      0.0      0.0        
 0.0      3.35786  0.0      0.0        
 0.0      0.0      3.03002  0.0        
 0.0      0.0      0.0      9.67579e-17

In [107]:
# decomposition is just as good if we ignore the 0 in sigma and reduce r by 1
norm(X - U[:,1:3]*diagm(σ[1:3])*(V[:,1:3])')

2.1374513381462106e-14

In [123]:
# for the skinny SVD, V'V is still the identity
V[:,1:3]'*V[:,1:3]

3×3 Array{Float64,2}:
  1.0          1.00519e-16  -1.42545e-16
  1.00519e-16  1.0           4.89305e-17
 -1.42545e-16  4.89305e-17   1.0        

In [124]:
# but for the skinny SVD, VV' is not the identity
V[:,1:3]*V[:,1:3]'

4×4 Array{Float64,2}:
  1.0          -2.18341e-16  -2.33192e-16   3.01518e-17
 -2.18341e-16   0.5           0.5           1.43457e-15
 -2.33192e-16   0.5           0.5          -1.61454e-15
  3.01518e-17   1.43457e-15  -1.61454e-15   1.0        

In [125]:
# Indeed, for the skinny SVD V must have a non-zero null space, 
# since it's a fat matrix
V[:,1:3]'

3×4 Array{Float64,2}:
 -0.69848   -0.502368   -0.502368    0.0858952
  0.246793  -0.0894302  -0.0894302   0.96078  
  0.671728  -0.489518   -0.489518   -0.263675 

# Use SVD to solve least squares regression

In [108]:
# form training data from noisy linear model
w♮ = randn(d)
y = X*w♮ + .1*randn(n);

In [109]:
# generate test set
Xtest = randn(10,d)
ytest = Xtest*w♮ + .1*randn(10);

In [110]:
# solve least squares problem to estimate w
# using full SVD
w = V*diagm(σ.^(-1))*U'*y

4-element Array{Float64,1}:
 -1.32102 
 -6.949e14
  6.949e14
  2.3981  

In [112]:
# solve least squares problem to estimate w
# using skinny SVD
w = V[:,1:3]*diagm(σ[1:3].^(-1))*(U[:,1:3])'*y

4-element Array{Float64,1}:
 -1.32102 
 -0.123872
 -0.123872
  0.284177

In [126]:
# how good is our estimate of w? - not very good...
norm(w - w♮)/norm(w♮)

0.2139721407141088

In [128]:
# how small mean square error? - quite small!
# (around the same size as the noise level, .1)
mean((ytest - Xtest*w).^2)

0.12267942887783287

In [129]:
# let's use the shorthand
w_backslash = X \ y
norm(w_backslash - w)

4.802215920498286e-15